# 📈 Experiment 4: Scaling Experiments — Where Do TFMs Break?

This is arguably the **most important experiment** for enterprise adoption.

We progressively increase dataset size from 500 to 150K rows and observe:
- At what point does each TFM hit its hard limit and crash?
- Where does performance start degrading (soft limits)?
- How does inference time scale (linear? quadratic?)?
- At what size do GBDTs overtake TFMs?

---

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.loader import load_credit_dataset
from src.visualization.plots import plot_scaling_curves, set_style

RESULTS_DIR = Path('../results')
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

DATASET = 'give_me_credit'
set_style()

## 1. Define Row Count Schedule

We test at logarithmically-spaced intervals to capture behavior across scales.

In [ ]:
ROW_COUNTS = [500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000, 150_000]

# Known model limits for reference
MODEL_LIMITS = {
    'TabPFN-v1':         {'hard': 1_000,   'soft': 1_000},
    'TabPFN-v2':         {'hard': 10_000,  'soft': 10_000},
    'TabPFN-v2.5':       {'hard': 50_000,  'soft': 50_000},
    'Real-TabPFN-2.5':   {'hard': 50_000,  'soft': 50_000},
    'TabICL-v1.1':       {'hard': None,    'soft': 50_000},
    'TabICLv2':          {'hard': None,    'soft': 100_000},
    'Mitra':             {'hard': None,    'soft': 10_000},
    'XGBoost-Default':   {'hard': None,    'soft': None},
    'CatBoost-Default':  {'hard': None,    'soft': None},
    'LightGBM-Default':  {'hard': None,    'soft': None},
}

print('Scaling schedule:', ROW_COUNTS)
print(f'\nModel hard limits (rows):')
for name, lim in MODEL_LIMITS.items():
    h = f"{lim['hard']:,}" if lim['hard'] else 'None'
    s = f"{lim['soft']:,}" if lim['soft'] else 'None'
    print(f"  {name:<20} hard={h:<10} soft={s}")

## 2. Run Scaling Benchmark

In [ ]:
from scripts.run_benchmark import get_zero_shot_models

models = get_zero_shot_models()
all_scaling_results = []

for n_rows in ROW_COUNTS:
    print(f"\n{'='*60}")
    print(f"  SCALING TEST: {n_rows:,} rows")
    print(f"{'='*60}")
    
    try:
        X_train, X_test, y_train, y_test = load_credit_dataset(
            DATASET, max_rows=n_rows, random_state=42
        )
    except Exception as e:
        print(f"  Cannot load {n_rows} rows: {e}")
        continue
    
    print(f"  Actual: {len(X_train):,} train / {len(X_test):,} test")
    
    for model in models:
        result = model.evaluate(
            X_train, y_train, X_test, y_test,
            DATASET, phase='scaling'
        )
        
        row = result.to_dict()
        row['n_rows_requested'] = n_rows
        all_scaling_results.append(row)
        
        status = '\u2705' if result.success else '\u274c'
        detail = f"AUC={result.auc_roc:.4f} Time={result.total_time:.2f}s" if result.success else result.error_message[:40]
        print(f"    {status} {model.name:<25} {detail}")

scaling_df = pd.DataFrame(all_scaling_results)
scaling_df.to_csv(RESULTS_DIR / f'scaling_{DATASET}.csv', index=False)
print(f"\n\u2705 Saved to {RESULTS_DIR}/scaling_{DATASET}.csv")

## 3. Scaling Curves

In [ ]:
fig = plot_scaling_curves(
    scaling_df, metric='auc_roc',
    title=f'Model Scaling Behavior: {DATASET}',
    save_path=str(FIGURES_DIR / f'scaling_{DATASET}.png')
)
plt.show()

## 4. Success/Failure Matrix

In [ ]:
# Create a heatmap of AUC-ROC across scales
pivot = scaling_df.pivot_table(
    index='model_name', columns='n_rows_requested',
    values='auc_roc', aggfunc='first'
)

fig, ax = plt.subplots(figsize=(14, max(6, len(pivot) * 0.5)))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0.5, vmax=1.0)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{c:,}" for c in pivot.columns], rotation=45)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('Number of Training Rows')
ax.set_title('AUC-ROC Across Scales (gray = failed)', fontweight='bold', fontsize=13)

# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if np.isnan(val):
            ax.text(j, i, '\u274c', ha='center', va='center', fontsize=10)
        else:
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'scaling_heatmap_{DATASET}.png', bbox_inches='tight')
plt.show()

## 5. Timing Analysis

In [ ]:
successful = scaling_df[scaling_df['success'] == True].copy()

fig, ax = plt.subplots(figsize=(12, 6))
for model in successful['model_name'].unique():
    mdf = successful[successful['model_name'] == model].sort_values('n_rows_requested')
    ax.plot(mdf['n_rows_requested'], mdf['total_time'], marker='o', label=model, linewidth=2)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Training Rows (log)')
ax.set_ylabel('Total Time in seconds (log)')
ax.set_title('Inference Time Scaling', fontweight='bold')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'timing_scaling_{DATASET}.png', bbox_inches='tight')
plt.show()

## 6. Key Findings Template

Fill in after running:

| Finding | Details |
|---------|--------|
| TabPFN v2 stops working at | ___ rows |
| TabPFN v2.5 stops working at | ___ rows |
| TabICLv2 stops working at | ___ rows |
| GBDTs overtake TFMs at | ~___ rows |
| Best model under 5K rows | ___ |
| Best model under 50K rows | ___ |
| Best model at 100K+ rows | ___ |
| Fastest model overall | ___ |
| TFM with best time/accuracy ratio | ___ |